In [ ]:
## ==============================================================================
## SO5H — Raspberry Pi 5 Edge Inference Preparation
## ==============================================================================

# 1. Google Drive

In [3]:
# =====================================================================
# Mount Drive and create  output directory
# =====================================================================
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

#  directory
SO5H_DIR = Path("/content/drive/MyDrive/TFM/SO5H_RPi_Demo")
SO5H_DIR.mkdir(parents=True, exist_ok=True)

MODELS_DIR = SO5H_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

NPY_DIR = SO5H_DIR / "test_npy"
NPY_DIR.mkdir(exist_ok=True)

# source paths for pre-trained INT8 models
SO5_DIR = Path("/content/drive/MyDrive/TFM/UNet_Light_CloudSEN12_B16/so5_deployment_v2")

print(f"SO5H output:  {SO5H_DIR}")
print(f"SO5 source:  {SO5_DIR}")
print(f"SO5 exists:  {SO5_DIR.exists()}")
!df -h /content/drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SO5H output:  /content/drive/MyDrive/TFM/SO5H_RPi_Demo
SO5 source:  /content/drive/MyDrive/TFM/UNet_Light_CloudSEN12_B16/so5_deployment_v2
SO5 exists:  True
Filesystem      Size  Used Avail Use% Mounted on
drive           100G   21G   80G  21% /content/drive


# 2. Download CloudSEN12+ Dataset

In [4]:
from huggingface_hub import snapshot_download

DATA_DIR = Path("/content/cloudsen12_high")

print("Ensuring CloudSEN12+ dataset is available ...")

snapshot_download(
    repo_id="csaybar/CloudSEN12-high",
    repo_type="dataset",
    allow_patterns=["*/L1C_*.dat", "*manual_hq*"],
    local_dir=str(DATA_DIR),
)

print("Dataset ready!")
for split in ["train", "val", "test"]:
    path = DATA_DIR / split
    n = len(list(path.glob("*.dat")))
    print(f"  {split}: {n} files")

!df -h /content

Ensuring CloudSEN12+ dataset is available ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]

Dataset ready!
  train: 14 files
  val: 14 files
  test: 14 files
Filesystem      Size  Used Avail Use% Mounted on
overlay         226G   87G  140G  39% /


# 3.Configuration

In [6]:
# =====================================================================
# Dataset parameters
# =====================================================================
import numpy as np

DATA_ROOT = Path("/content/cloudsen12_high")
TEST_DIR  = DATA_ROOT / "test"

NUM_S2_BANDS  = 13
IGNORE_LABEL  = 99
TARGET_SIZE   = 512
DIVISOR       = 10000.0
CLOUD_CLASSES = {1, 2}
SPLIT_SIZES   = {"train": 8490, "val": 535, "test": 975}

L1C_BAND_FILES = [
    "L1C_B1.dat", "L1C_B2.dat", "L1C_B3.dat", "L1C_B4.dat",
    "L1C_B5.dat", "L1C_B6.dat", "L1C_B7.dat", "L1C_B8.dat",
    "L1C_B8A.dat", "L1C_B9.dat", "L1C_B10.dat", "L1C_B11.dat",
    "L1C_B12.dat",
]

print(f"Test set: {SPLIT_SIZES['test']} patches, {NUM_S2_BANDS} bands, {TARGET_SIZE}x{TARGET_SIZE}")

Test set: 975 patches, 13 bands, 512x512


# 4. Export Test Set as .npy

Preparation of 975 test patches appling the same preprocessing as the training pipeline and saves each patch
as a pair of .npy files.

In [7]:
# =====================================================================
# Export test set to individual .npy files
# =====================================================================
from tqdm.auto import tqdm

def load_memmap_split(split_dir):
    """Open memmap arrays for all 13 L1C bands + label."""
    split_name = Path(split_dir).name
    n = SPLIT_SIZES[split_name]
    shape = (n, TARGET_SIZE, TARGET_SIZE)
    bands = []
    for fname in L1C_BAND_FILES:
        fpath = Path(split_dir) / fname
        bands.append(np.memmap(str(fpath), dtype="int16", mode="r", shape=shape))
    label = np.memmap(
        str(Path(split_dir) / "LABEL_manual_hq.dat"),
        dtype="int8", mode="r", shape=shape,
    )
    return bands, label, n

# load test split
bands, label, n_test = load_memmap_split(TEST_DIR)
print(f"Loaded {n_test} test patches from memmap")

for i in tqdm(range(n_test), desc="Exporting .npy"):
    # image: stack 13 bands, normalise
    x = np.stack([band[i] for band in bands], axis=-1)
    # convert to float32
    x = x.astype(np.float32)
    # normalize reflectance
    x = x / DIVISOR
    # values between 0 and 1
    x = np.clip(x, 0.0, 1.0)

    # label: binary mapping + ignore
    y_raw = label[i].astype(np.float32)
    y_int = y_raw.astype(int)
    is_cloud = np.isin(y_int, list(CLOUD_CLASSES))
    y_bin = np.where(is_cloud, 1.0, 0.0)
    y_bin = np.where(y_int == IGNORE_LABEL, float(IGNORE_LABEL), y_bin)

    # zero-padded pixels -> ignore
    all_zero = np.all(x == 0.0, axis=-1)
    y_bin = np.where(all_zero, float(IGNORE_LABEL), y_bin)

    # save
    np.save(NPY_DIR / f"x_{i:04d}.npy", x)               # (512,512,13) float32
    np.save(NPY_DIR / f"y_{i:04d}.npy", y_bin[..., None]) # (512,512,1)  float32

# verify
n_x = len(list(NPY_DIR.glob("x_*.npy")))
n_y = len(list(NPY_DIR.glob("y_*.npy")))
size_gb = sum(f.stat().st_size for f in NPY_DIR.iterdir()) / 1e9
print(f"\nExported: {n_x} images, {n_y} labels")
print(f"Total size: {size_gb:.1f} GB")

Loaded 975 test patches from memmap


Exporting .npy:   0%|          | 0/975 [00:00<?, ?it/s]


Exported: 975 images, 975 labels
Total size: 15.3 GB
